In [9]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# STEP 1: Load Raw File and Inspect Sheet Names
# -------------------------------------------------------------------
excel_file = 'SaaS Model vCurrent.xlsx'
xls = pd.ExcelFile(excel_file)
print("Available Sheets:", xls.sheet_names)

# -------------------------------------------------------------------
# STEP 2: Function to Process Wide SaaS Sheets into Long/Tidy Format
# -------------------------------------------------------------------
def process_saas_sheet(excel_path, sheet_name):
    # Read raw sheet without predefined headers
    raw_df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    
    # Row index 2 contains Years (2024, 2025...), Row index 3 contains Month Names/Dates
    years = raw_df.iloc[2, 6:].values
    months = raw_df.iloc[3, 6:].values
    
    # Construct proper Year-Month Datetime headers for columns from index 6 onwards
    dates = []
    for y, m in zip(years, months):
        if pd.isna(y) or pd.isna(m):
            dates.append(None)
        else:
            # Handle if month is already a timestamp or string
            m_str = str(m).split()[0] if isinstance(m, pd.Timestamp) else str(m).strip()
            dates.append(f"{int(y)}-{m_str}")
            
    # Clean KPI metric names column (usually located at column index 2)
    metric_names = raw_df.iloc[:, 2].ffill() # Forward fill categories if empty
    sub_metric_names = raw_df.iloc[:, 2]
    
    # Slice actual numeric data starting from row 6 onwards
    data_df = raw_df.iloc[5:, :].copy()
    
    # Assign Metric Names
    data_df['Metric'] = data_df[2].astype(str).str.strip()
    
    # Drop rows where Metric is NaN, empty, or header dividers
    data_df = data_df[data_df['Metric'].notna() & (data_df['Metric'] != 'nan') & (data_df['Metric'] != '')]
    
    # Identify date columns
    date_col_indices = [i for i, d in enumerate(dates) if d is not None]
    actual_date_names = [dates[i] for i in date_col_indices]
    actual_col_numbers = [i + 6 for i in date_col_indices]
    
    # Filter useful columns
    keep_cols = ['Metric'] + actual_col_numbers
    data_df = data_df[keep_cols]
    
    # Rename columns to Date Strings
    col_rename_dict = {num: name for num, name in zip(actual_col_numbers, actual_date_names)}
    data_df = data_df.rename(columns=col_rename_dict)
    
    # Melt/Unpivot from Wide Format to Long Format
    long_df = pd.melt(
        data_df, 
        id_vars=['Metric'], 
        var_name='Date_Str', 
        value_name='Value'
    )
    
    # Convert Value to numeric, replace invalid values/symbols with NaN
    long_df['Value'] = pd.to_numeric(long_df['Value'], errors='coerce')
    
    # Convert Date_Str to proper datetime format
    long_df['Date'] = pd.to_datetime(long_df['Date_Str'], format='%Y-%b', errors='coerce')
    long_df = long_df.dropna(subset=['Date', 'Value'])
    
    long_df['Sheet_Source'] = sheet_name
    return long_df[['Date', 'Sheet_Source', 'Metric', 'Value']]

# -------------------------------------------------------------------
# STEP 3: Clean and Merge Primary Sheets
# -------------------------------------------------------------------
sheets_to_process = ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']
cleaned_dataframes = []

for sheet in sheets_to_process:
    try:
        df_cleaned = process_saas_sheet(excel_file, sheet)
        cleaned_dataframes.append(df_cleaned)
        print(f" Successfully processed: {sheet} ({len(df_cleaned)} rows extracted)")
    except Exception as e:
        print(f" Error processing {sheet}: {e}")

# Combine all into one master long-format dataframe
master_df = pd.concat(cleaned_dataframes, ignore_index=True)

# Remove duplicates if any
master_df = master_df.drop_duplicates()



Available Sheets: ['Cover', 'Metrics Dashboard', 'Detailed Metrics', 'Financials >>>', 'Assumptions', 'Summary Financials', 'Detailed Financials', 'Subscribers & Revenue']
 Successfully processed: Subscribers & Revenue (0 rows extracted)
 Successfully processed: Detailed Metrics (0 rows extracted)
 Successfully processed: Summary Financials (0 rows extracted)


In [2]:
import pandas as pd

excel_path = 'SaaS Model vCurrent.xlsx'
df = pd.read_excel(excel_path, sheet_name='Detailed Metrics', header=None)

# Let's inspect rows 0 to 10 and columns 0 to 10
print("Rows 0 to 8:")
print(df.iloc[0:9, 0:10])

Rows 0 to 8:
     0    1            2   3            4   5                    6  \
0  NaN  NaN          NaN NaN          NaN NaN                  NaN   
1  NaN  NaN          NaN NaN          NaN NaN                  NaN   
2  NaN  NaN          NaN NaN          NaN NaN                  NaN   
3  NaN  NaN          NaN NaN          NaN NaN                 2024   
4  NaN  NaN          NaN NaN          NaN NaN                    1   
5  NaN  NaN          NaN NaN          NaN NaN  2024-01-01 00:00:00   
6  NaN  NaN          NaN NaN  Assumptions NaN                    1   
7  NaN  NaN          NaN NaN          NaN NaN                  NaN   
8    x    x  P&L Metrics NaN          NaN NaN                  NaN   

                     7                    8                    9  
0                  NaN                  NaN                  NaN  
1                  NaN                  NaN                  NaN  
2                  NaN                  NaN                  NaN  
3                 

In [3]:
# Let's inspect rows 8 to 25 to see how metrics are structured
print("Rows 8 to 25:")
print(df.iloc[8:25, [0, 1, 2, 4, 6, 7]])

Rows 8 to 25:
      0    1                             2    4       6         7
8     x    x                   P&L Metrics  NaN     NaN       NaN
9   NaN  NaN                           NaN  NaN     NaN       NaN
10  NaN  NaN  Monthly Plan Monthly Revenue  NaN    2700      5250
11  NaN  NaN                      % growth  NaN     NaN  0.944444
12  NaN  NaN                           NaN  NaN     NaN       NaN
13  NaN  NaN   Annual Plan Monthly Revenue  NaN  1512.5      3135
14  NaN  NaN                      % growth  NaN     NaN  1.072727
15  NaN  NaN                           NaN  NaN     NaN       NaN
16  NaN    x                           MRR  NaN  4212.5      8385
17  NaN  NaN                      % growth  NaN     NaN  0.990504
18  NaN  NaN                           NaN  NaN     NaN       NaN
19  NaN  NaN   Monthly Plan Annual Revenue  NaN   32400     63000
20  NaN  NaN                      % growth  NaN     NaN  0.944444
21  NaN  NaN                           NaN  NaN     NaN       

In [4]:
# Write dynamic parser and test it live
def parse_saas_sheet(excel_path, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    
    # Row 5 has the actual Datetime objects starting at column index 6
    date_row = df.iloc[5, 6:]
    
    # Identify valid date columns
    valid_cols = []
    dates = []
    for col_idx, val in date_row.items():
        if pd.notna(val) and (isinstance(val, pd.Timestamp) or isinstance(val, str)):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
                
    # Metric names are in Column 2 (index 2) starting from row 8
    metrics_series = df.iloc[8:, 2]
    
    data_rows = []
    for row_idx, metric in metrics_series.items():
        if pd.isna(metric) or str(metric).strip() == '':
            continue
        metric_name = str(metric).strip()
        
        # Get values for this row across valid date columns
        row_values = df.iloc[row_idx, valid_cols].values
        
        for dt, val in zip(dates, row_values):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({
                        'Date': dt,
                        'Sheet': sheet_name,
                        'Metric': metric_name,
                        'Value': num_val
                    })
                    
    return pd.DataFrame(data_rows)

# Test on sheets
dfs = []
for s in ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']:
    parsed = parse_saas_sheet(excel_path, s)
    dfs.append(parsed)
    print(f"Sheet '{s}': {len(parsed)} rows extracted.")

master = pd.concat(dfs, ignore_index=True)
print("Total records extracted:", len(master))
print(master.head())

Sheet 'Subscribers & Revenue': 0 rows extracted.
Sheet 'Detailed Metrics': 0 rows extracted.
Sheet 'Summary Financials': 0 rows extracted.
Total records extracted: 0
Empty DataFrame
Columns: []
Index: []


In [5]:
# Debug why date_row or rows didn't match
df_sub = pd.read_excel(excel_path, sheet_name='Subscribers & Revenue', header=None)
print("Subscribers & Revenue Row 0 to 10:")
print(df_sub.iloc[0:10, 0:10])

Subscribers & Revenue Row 0 to 10:
     0    1            2   3            4   5                    6  \
0  NaN  NaN          NaN NaN          NaN NaN                  NaN   
1  NaN  NaN          NaN NaN          NaN NaN                  NaN   
2  NaN  NaN          NaN NaN          NaN NaN                  NaN   
3  NaN  NaN          NaN NaN          NaN NaN                 2024   
4  NaN  NaN          NaN NaN          NaN NaN                    1   
5  NaN  NaN          NaN NaN          NaN NaN  2024-01-01 00:00:00   
6  NaN  NaN          NaN NaN  Assumptions NaN                    1   
7  NaN  NaN          NaN NaN          NaN NaN                  NaN   
8    x    x  Subscribers NaN          NaN NaN                  NaN   
9  NaN  NaN          NaN NaN          NaN NaN                  NaN   

                     7                    8                    9  
0                  NaN                  NaN                  NaN  
1                  NaN                  NaN                 

In [6]:
# Check types of row 5 elements
row5 = df_sub.iloc[5, 6:]
print("Row 5 types:", [type(x) for x in row5[:5]])
print("Row 5 values:", row5[:5].values)

Row 5 types: [<class 'datetime.datetime'>, <class 'datetime.datetime'>, <class 'datetime.datetime'>, <class 'datetime.datetime'>, <class 'datetime.datetime'>]
Row 5 values: [datetime.datetime(2024, 1, 1, 0, 0) datetime.datetime(2024, 2, 1, 0, 0)
 datetime.datetime(2024, 3, 1, 0, 0) datetime.datetime(2024, 4, 1, 0, 0)
 datetime.datetime(2024, 5, 1, 0, 0)]


In [7]:
# Fix parser function
def parse_saas_sheet_fixed(excel_path, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    
    # Dates are in Row index 5 starting from Col index 6
    date_series = df.iloc[5, 6:]
    
    valid_cols = []
    dates = []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
                
    data_rows = []
    # Loop over metric rows from index 8 onwards
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == '':
            continue
        metric_name = str(metric_val).strip()
        
        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({
                        'Date': dt,
                        'Sheet': sheet_name,
                        'Metric': metric_name,
                        'Value': num_val
                    })
                    
    return pd.DataFrame(data_rows)

dfs = []
for s in ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']:
    parsed = parse_saas_sheet_fixed(excel_path, s)
    dfs.append(parsed)
    print(f"Sheet '{s}': {len(parsed)} rows extracted.")

master = pd.concat(dfs, ignore_index=True)
print("Total records extracted:", len(master))
print(master.head())
print("Date range:", master['Date'].min().strftime('%Y-%m'), "to", master['Date'].max().strftime('%Y-%m'))

Sheet 'Subscribers & Revenue': 2278 rows extracted.
Sheet 'Detailed Metrics': 2980 rows extracted.
Sheet 'Summary Financials': 3179 rows extracted.
Total records extracted: 8437
        Date                  Sheet                   Metric  Value
0 2024-01-01  Subscribers & Revenue  New Monthly Subscribers   90.0
1 2024-02-01  Subscribers & Revenue  New Monthly Subscribers   99.0
2 2024-03-01  Subscribers & Revenue  New Monthly Subscribers  109.0
3 2024-04-01  Subscribers & Revenue  New Monthly Subscribers  120.0
4 2024-05-01  Subscribers & Revenue  New Monthly Subscribers  132.0
Date range: 2024-01 to 2028-12


In [10]:
# Let's generate the complete working script that parses all sheets properly
code_solution = """import pandas as pd

# File path
excel_file = 'SaaS Model vCurrent.xlsx'

def parse_saas_sheet(excel_path, sheet_name):
    # Read sheet without header
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    
    # Dates are in Row index 5, starting from Column index 6
    date_series = df.iloc[5, 6:]
    
    valid_cols = []
    dates = []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
                
    data_rows = []
    # Metrics start from Row index 8 onwards in Column index 2
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == '':
            continue
        metric_name = str(metric_val).strip()
        
        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({
                        'Date': dt,
                        'Sheet_Source': sheet_name,
                        'Metric': metric_name,
                        'Value': num_val
                    })
                    
    return pd.DataFrame(data_rows)

# Sheets to extract data from
sheets_to_process = ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']
cleaned_dfs = []

for sheet in sheets_to_process:
    try:
        df_sheet = parse_saas_sheet(excel_file, sheet)
        cleaned_dfs.append(df_sheet)
        print(f"Successfully processed: '{sheet}' ({len(df_sheet)} rows extracted)")
    except Exception as e:
        print(f"Error processing sheet '{sheet}': {e}")

# Combine into master dataframe
master_df = pd.concat(cleaned_dfs, ignore_index=True).drop_duplicates()

# Dataset Summary Check
print("\\n--- Master Dataset Summary ---")
print("Total Records:", len(master_df))

if not master_df.empty and master_df['Date'].notna().any():
    min_date = master_df['Date'].min().strftime('%Y-%m')
    max_date = master_df['Date'].max().strftime('%Y-%m')
    print(f"Date Range: {min_date} to {max_date}")
    print("\\nSample Extracted Metrics:")
    print(master_df['Metric'].unique()[:10])
    
    # Save clean dataset
    output_csv = 'Cleaned_SaaS_Metrics.csv'
    master_df.to_csv(output_csv, index=False)
    print(f"\\n Cleaned data saved successfully to '{output_csv}'!")
"""

print("Code ready")

Code ready


In [11]:
import pandas as pd

# File name setup
excel_file = "SaaS Model vCurrent.xlsx"


def parse_saas_sheet(excel_path, sheet_name):
    # Read raw sheet
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    # Dates are in Row index 5, starting from Column index 6
    date_series = df.iloc[5, 6:]

    valid_cols = []
    dates = []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors="coerce")
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)

    data_rows = []
    # Metrics start from Row index 8 onwards in Column index 2
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == "":
            continue
        metric_name = str(metric_val).strip()

        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors="coerce")
                if pd.notna(num_val):
                    data_rows.append({
                        "Date": dt,
                        "Sheet_Source": sheet_name,
                        "Metric": metric_name,
                        "Value": num_val,
                    })

    return pd.DataFrame(data_rows)


# Sheets to extract data from
sheets_to_process = [
    "Subscribers & Revenue",
    "Detailed Metrics",
    "Summary Financials",
]
cleaned_dfs = []

for sheet in sheets_to_process:
    try:
        df_sheet = parse_saas_sheet(excel_file, sheet)
        cleaned_dfs.append(df_sheet)
        print(
            f"Successfully processed: '{sheet}' ({len(df_sheet)} rows extracted)"
        )
    except Exception as e:
        print(f"Error processing sheet '{sheet}': {e}")

# Combine into master dataframe
master_df = pd.concat(cleaned_dfs, ignore_index=True).drop_duplicates()

# Dataset Summary Check
print("\n--- Master Dataset Summary ---")
print("Total Records:", len(master_df))

if not master_df.empty and master_df["Date"].notna().any():
    min_date = master_df["Date"].min().strftime("%Y-%m")
    max_date = master_df["Date"].max().strftime("%Y-%m")
    print(f"Date Range: {min_date} to {max_date}")
    print("\nSample Extracted Metrics:")
    print(master_df["Metric"].unique()[:10])

    # Save clean dataset
    output_csv = "Cleaned_SaaS_Metrics.csv"
    master_df.to_csv(output_csv, index=False)
    print(f"\n Cleaned data saved successfully to '{output_csv}'!")

Successfully processed: 'Subscribers & Revenue' (2278 rows extracted)
Successfully processed: 'Detailed Metrics' (2980 rows extracted)
Successfully processed: 'Summary Financials' (3179 rows extracted)

--- Master Dataset Summary ---
Total Records: 7315
Date Range: 2024-01 to 2028-12

Sample Extracted Metrics:
['New Monthly Subscribers' '% growth' 'Previous Month Active Subscribers'
 '(+) New Subscribers' '(–) Churned Subscribers'
 'Active Monthly Subscribers' 'New Annual Subscribers'
 'Active Annual Subscribers' 'Memo: Renewals' 'New Subscribers']

 Cleaned data saved successfully to 'Cleaned_SaaS_Metrics.csv'!


In [12]:
import pandas as pd

master_df = pd.read_csv('Cleaned_SaaS_Metrics.csv')
print("Extracted metrics list:")
print(master_df['Metric'].value_counts())

Extracted metrics list:
Metric
% growth                      495
ARR                           180
New Monthly Subscribers       120
Cash from Operations          120
MRR                           120
                             ... 
Net New ARR                    59
Cash Balance December 2025      1
Funds to Raise                  1
Runway Month                    1
Memo: Fundraising Start         1
Name: count, Length: 102, dtype: int64


In [13]:
# Re-generate master_df internally to display exact available key metrics
excel_file = 'SaaS Model vCurrent.xlsx'

def parse_saas_sheet(excel_path, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    date_series = df.iloc[5, 6:]
    valid_cols, dates = [], []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
    data_rows = []
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == '':
            continue
        metric_name = str(metric_val).strip()
        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({'Date': dt, 'Sheet_Source': sheet_name, 'Metric': metric_name, 'Value': num_val})
    return pd.DataFrame(data_rows)

dfs = [parse_saas_sheet(excel_file, s) for s in ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']]
master_df = pd.concat(dfs, ignore_index=True).drop_duplicates()
print("Top 20 Key Metrics:")
print(master_df['Metric'].unique()[:20])

Top 20 Key Metrics:
['New Monthly Subscribers' '% growth' 'Previous Month Active Subscribers'
 '(+) New Subscribers' '(–) Churned Subscribers'
 'Active Monthly Subscribers' 'New Annual Subscribers'
 'Active Annual Subscribers' 'Memo: Renewals' 'New Subscribers'
 'Total Active Subscribers' 'Monthly Plan Price / month'
 'Monthly Plan Bookings' '(+) Renewals' 'Total Annual Subscribers'
 'Annual Plan Price / year' 'Annual Plan Bookings' 'Total Bookings'
 'Monthly Plan Revenue' 'Annual Plan Monthly Revenue']


In [14]:
import pandas as pd
import matplotlib.pyplot as plt

# Re-run extraction internally to create dataframe for chart generation
excel_file = 'SaaS Model vCurrent.xlsx'

def parse_saas_sheet(excel_path, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    date_series = df.iloc[5, 6:]
    valid_cols, dates = [], []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
    data_rows = []
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == '':
            continue
        metric_name = str(metric_val).strip()
        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({'Date': dt, 'Sheet_Source': sheet_name, 'Metric': metric_name, 'Value': num_val})
    return pd.DataFrame(data_rows)

dfs = [parse_saas_sheet(excel_file, s) for s in ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']]
master_df = pd.concat(dfs, ignore_index=True).drop_duplicates()

# Generate visual sample for MRR Trend
mrr_df = master_df[master_df['Metric'] == 'MRR'].sort_values('Date')

plt.figure(figsize=(10, 4.5))
plt.plot(mrr_df['Date'], mrr_df['Value'], color='#1f77b4', linewidth=2.5, label='MRR ($)')
plt.title('SaaS Monthly Recurring Revenue (MRR) Growth Trend (2024 - 2028)', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('MRR ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('mrr_trend_preview.png', dpi=300)
plt.close()

print("Plot saved successfully")

Plot saved successfully


In [15]:
import plotly.express as px
import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="SaaS KPI Executive Dashboard", layout="wide"
)

# Load cleaned CSV data
df = pd.read_csv("Cleaned_SaaS_Metrics.csv")
df["Date"] = pd.to_datetime(df["Date"])

st.title("🚀 SaaS Executive KPI Dashboard")

# Top KPI Summary Cards
col1, col2, col3 = st.columns(3)
latest_mrr = df[df["Metric"] == "MRR"].iloc[-1]["Value"]
latest_subs = df[df["Metric"] == "Total Active Subscribers"].iloc[-1]["Value"]

col1.metric("Current MRR", f"${latest_mrr:,.0f}")
col2.metric("Current ARR", f"${latest_mrr*12:,.0f}")
col3.metric("Active Subscribers", f"{int(latest_subs):,}")

st.markdown("---")

# Visual 1: MRR Growth Trend
mrr_df = df[df["Metric"] == "MRR"]
fig_mrr = px.line(
    mrr_df,
    x="Date",
    y="Value",
    title="Monthly Recurring Revenue (MRR) Growth",
)
st.plotly_chart(fig_mrr, use_container_width=True)

2026-08-25 10:36:30.115 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 10:36:30.147 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 10:36:30.413 
  command:

    streamlit run c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-25 10:36:30.414 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 10:36:30.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 10:36:30.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 10:36:30.416 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

In [16]:
# Complete Dashboard App Code Generation
streamlit_app_code = """import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as px_go

st.set_page_config(
    page_title="SaaS Executive KPI Dashboard",
    page_icon="📊",
    layout="wide"
)

# Load Cleaned Data
@st.cache_data
def load_data():
    df = pd.read_csv("Cleaned_SaaS_Metrics.csv")
    df['Date'] = pd.to_datetime(df['Date'])
    return df

df = load_data()

# Sidebar Filters
st.sidebar.header(" Filter Options")
years = sorted(df['Date'].dt.year.unique())
selected_year = st.sidebar.multiselect("Select Years", options=years, default=years)

# Filter Data
filtered_df = df[df['Date'].dt.year.isin(selected_year)]

st.title("📊 SaaS Revenue & Retention Executive Dashboard")
st.markdown("---")

# 1. KPI Cards Row
col1, col2, col3, col4 = st.columns(4)

mrr_df = filtered_df[filtered_df['Metric'] == 'MRR']
latest_mrr = mrr_df.iloc[-1]['Value'] if not mrr_df.empty else 0
latest_arr = latest_mrr * 12

subs_df = filtered_df[filtered_df['Metric'] == 'Total Active Subscribers']
latest_subs = subs_df.iloc[-1]['Value'] if not subs_df.empty else 0
arpu = (latest_mrr / latest_subs) if latest_subs > 0 else 0

col1.metric("Current MRR", f"${latest_mrr:,.0f}")
col2.metric("Current ARR", f"${latest_arr:,.0f}")
col3.metric("Active Subscribers", f"{int(latest_subs):,}")
col4.metric("ARPU (Avg Rev / User)", f"${arpu:,.2f}")

st.markdown("###")

# 2. Charts Row 1: MRR Growth & Revenue Breakdown
chart_col1, chart_col2 = st.columns([2, 1])

with chart_col1:
    st.subheader("Monthly Recurring Revenue (MRR) Trend")
    fig_mrr = px.line(mrr_df, x='Date', y='Value', markers=True, title="MRR Trajectory Over Time")
    fig_mrr.update_traces(line_color='#1f77b4', line_width=3)
    st.plotly_chart(fig_mrr, use_container_width=True)

with chart_col2:
    st.subheader("Revenue Split by Plan")
    rev_metrics = ['Monthly Plan Revenue', 'Annual Plan Monthly Revenue']
    plan_df = filtered_df[filtered_df['Metric'].isin(rev_metrics)].groupby('Metric')['Value'].sum().reset_index()
    fig_pie = px.pie(plan_df, values='Value', names='Metric', hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
    st.plotly_chart(fig_pie, use_container_width=True)

# 3. Charts Row 2: Customer Acquisition vs Churn
st.subheader("Customer Growth: New vs Churned Subscribers")
sub_metrics = ['(+) New Subscribers', '(–) Churned Subscribers']
sub_flow_df = filtered_df[filtered_df['Metric'].isin(sub_metrics)]
fig_bar = px.bar(sub_flow_df, x='Date', y='Value', color='Metric', barmode='group',
                 color_discrete_map={'(+) New Subscribers': '#2ca02c', '(–) Churned Subscribers': '#d62728'})
st.plotly_chart(fig_bar, use_container_width=True)

# 4. Raw Data Preview Table
with st.expander(" Data Explorer Table"):
    st.dataframe(filtered_df, use_container_width=True)
"""

with open("app.py", "w") as f:
    f.write(streamlit_app_code)

print("Streamlit app code written to app.py successfully.")

Streamlit app code written to app.py successfully.


In [22]:
# Final sanity check: Verify Cleaned_SaaS_Metrics.csv output completely
import pandas as pd

excel_file = 'SaaS Model vCurrent.xlsx'

def parse_saas_sheet(excel_path, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    date_series = df.iloc[5, 6:]
    valid_cols, dates = [], []
    for col_idx, val in date_series.items():
        if pd.notna(val):
            dt = pd.to_datetime(val, errors='coerce')
            if pd.notna(dt):
                valid_cols.append(col_idx)
                dates.append(dt)
    data_rows = []
    for row_idx in range(8, len(df)):
        metric_val = df.iloc[row_idx, 2]
        if pd.isna(metric_val) or str(metric_val).strip() == '':
            continue
        metric_name = str(metric_val).strip()
        row_vals = df.iloc[row_idx, valid_cols].values
        for dt, val in zip(dates, row_vals):
            if pd.notna(val):
                num_val = pd.to_numeric(val, errors='coerce')
                if pd.notna(num_val):
                    data_rows.append({'Date': dt, 'Sheet_Source': sheet_name, 'Metric': metric_name, 'Value': num_val})
    return pd.DataFrame(data_rows)

dfs = [parse_saas_sheet(excel_file, s) for s in ['Subscribers & Revenue', 'Detailed Metrics', 'Summary Financials']]
master_df = pd.concat(dfs, ignore_index=True).drop_duplicates()
master_df.to_csv('Cleaned_SaaS_Metrics.csv', index=False)

print("Master DataFrame shape:", master_df.shape)
print("Saved Cleaned_SaaS_Metrics.csv successfully.")

Master DataFrame shape: (7315, 4)
Saved Cleaned_SaaS_Metrics.csv successfully.


In [ ]:
from urllib.parse import quote_plus
import pandas as pd
from sqlalchemy import create_engine

# 1. Database Credentials
user = "*****"
password = "*****"  
host = "*****"
port = "*****"
db_name = "saas_kpi_dashboard"

# 2. Safe Password Encoding (@ character fix ke liye)
safe_password = quote_plus(password)

# 3. Create SQLAlchemy Connection Engine
engine_url = f"postgresql://{user}:{safe_password}@{host}:{port}/{db_name}"
engine = create_engine(engine_url)

# 4. Load CSV & Export to PostgreSQL Database
df = pd.read_csv("Cleaned_SaaS_Metrics.csv")
df.to_sql("saas_metrics", engine, if_exists="replace", index=False)

print(
    " Successfully connected to PostgreSQL and imported all 7,300+ rows into saas_metrics!"
)

 Successfully connected to PostgreSQL and imported all 7,300+ rows into saas_metrics!
